# M08. groupby + 집계

> 📌 **언제 필요한가**  
> 그룹별로 합/평균/개수 등을 구할 때.  
> 예: "시도별 졸업자 총합", "학년별 평균 점수", "연도별 인구 변화"

## 이 모듈에서 배울 것

- `groupby` 기본 — 분할(split) → 적용(apply) → 결합(combine)
- 여러 집계 함수 한 번에 (`agg`)
- 여러 컬럼 한꺼번에 그룹화
- 결과 정리 (`reset_index`)

---

## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.
> 받는 곳 링크를 누르면 바로 받으러 갈 수 있어요.

- `한국교육개발원_시도 시군구별 졸업자 진학자 진학률_20240401.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15053808/fileData.do)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. groupby 기본 사고방식

> **"OO별로 ZZ를 구하라"** 형태의 질문이면 groupby예요.
> 
> - "**시도별** 졸업자 **총합**" → `groupby('시도')['졸업자'].sum()`
> - "**연도별** 평균 점수" → `groupby('연도')['점수'].mean()`
> - "**성별** 인구 **개수**" → `groupby('성별').size()`


In [ ]:
import pandas as pd

graduate = pd.read_csv('data/한국교육개발원 시도 시군구별 졸업자 진학자 진학률_20240401.csv', encoding='cp949')
print(f"shape: {graduate.shape}")
graduate.head()


## 2. 가장 단순한 groupby


In [ ]:
# 시도별 졸업자 총합
graduate.groupby('시도')['졸업자'].sum()


In [ ]:
# 시도별 졸업자 평균
graduate.groupby('시도')['졸업자'].mean()


In [ ]:
# 시도별 행 개수 (시군구 수)
graduate.groupby('시도').size()


## 3. 여러 컬럼 한 번에 — `agg`


In [ ]:
# 졸업자 합 + 진학자 합 한 번에
graduate.groupby('시도')[['졸업자', '진학자']].sum().head()


In [ ]:
# 다양한 집계 함수 한 번에 — agg 사용
graduate.groupby('시도').agg({
    '졸업자': ['sum', 'mean'],
    '진학자': 'sum'
}).head()


## 4. 여러 컬럼으로 그룹화 (다중 키)


In [ ]:
# 시도 + 연도로 그룹화
graduate.groupby(['시도', '연도'])['졸업자'].sum().head(10)


결과가 **멀티인덱스**가 됨. 보통은 `reset_index()`로 평탄화:


In [ ]:
# reset_index로 평탄한 DataFrame 만들기
graduate.groupby(['시도', '연도'])['졸업자'].sum().reset_index().head(10)


## 5. 정렬해서 보기 (`sort_values`와 조합)


In [ ]:
# 졸업자가 많은 시도 순으로 정렬
result = graduate.groupby('시도')['졸업자'].sum().reset_index()
result = result.sort_values('졸업자', ascending=False)
result.head(10)


## 6. 본인 데이터에 적용해보기 ✏️


In [ ]:
# my_df = pd.read_csv('파일.csv', encoding='cp949')
# 
# # "OO별 ZZ" 형태로 질문 만들기
# # 예: "지역별 평균 매출"
# result = my_df.groupby('지역')['매출'].mean()
# 
# # 여러 컬럼 한 번에
# result = my_df.groupby('지역').agg({
#     '매출': 'sum',
#     '직원수': 'mean'
# })


## 7. ⚠️ 함정 / 주의사항

### 7.1 groupby 결과는 인덱스가 그룹 컬럼이 됨
DataFrame처럼 다루려면 `reset_index()` 추천:
```python
df.groupby('시도').sum().reset_index()
```

### 7.2 결측치는 자동으로 제외됨
`sum()`, `mean()` 등은 NaN을 무시. 명시적으로 포함하려면 `dropna=False`.

### 7.3 빈 그룹이 있을 수 있음
카테고리 타입은 안 나타나는 그룹도 있을 수 있음. `observed=True` 옵션 검토.

### 7.4 정렬은 그룹 결과만
`sort_values`는 group 결과의 정렬일 뿐, 원본 정렬에 영향 없음.


## 8. 📚 더 알아보기

- `transform` — 그룹 통계를 원본 행에 매핑
- `filter` — 조건 만족하는 그룹만 남기기
- `apply` — 그룹마다 임의 함수 적용
- `pivot_table` — groupby + 멀티인덱스 자동
